# CodeOrbit real-agent test - Kaggle (unattended, two GPUs)

A local model localises each SWE-bench issue twice: with the usual agent tools
(grep, read_file, list_dir), and with those plus CodeOrbit's MCP tools. Two
models run at once, one per GPU.

### Once, in the right-hand panel (Settings)
- **Accelerator:** GPU T4 x2
- **Internet:** On

### To run
1. Check the settings cell below.
2. **Save Version -> Save & Run All (Commit) -> Save.** You can close the browser.
3. Later: open the notebook -> latest version -> **Output** -> download
   `codeorbit-agent-runs.zip`.

### If it stops early
Each model stops itself after `MAX_HOURS`. To continue: **Add Input -> Your Work
-> this notebook**, then put each model's run folder (shown under
`/kaggle/input/.../codeorbit-agent-runs/`) into `RESUME` and Save & Run All
again. Finished tasks are skipped.

## Settings

In [ ]:
MODELS = ['qwen3:8b', 'llama3.1:8b']   # one per GPU, run side by side
MODE = 'full'      # 'full'    - all 300 SWE-bench Lite tasks (the result to report)
                   # 'heldout' - the 240 tasks not used to tune fusion
                   # 'pilot'   - 6 small tasks: check everything works first
SAMPLE = 0         # 0 = every selected task; e.g. 120 = a seeded random 120 of them
MAX_HOURS = 11     # stop cleanly before Kaggle's 12 h limit
RESUME = {}        # to continue: {'qwen3:8b': '/kaggle/input/.../codeorbit-agent-runs/<run>', ...}

## 1. GPUs, get CodeOrbit

In [ ]:
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv

RUNS = '/kaggle/working/codeorbit-agent-runs'
SRC = '/tmp/codeorbit'
os.makedirs(RUNS, exist_ok=True)

%cd /tmp
!rm -rf {SRC}
!git clone -q --depth 1 https://github.com/Anilll-coder/codeorbit.git {SRC}
%cd {SRC}
!git log -1 --format='%h %s'
!pip install -q -e . tokenizers

## 2. Ollama with both models

One server; each model loads onto its own GPU. `ollama ps` at the end should
list both models (and the embedding model) at 100% GPU.

In [ ]:
!apt-get -qq install -y pciutils lshw zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, requests
env = dict(os.environ, OLLAMA_MAX_LOADED_MODELS='3', OLLAMA_NUM_PARALLEL='1',
           OLLAMA_KEEP_ALIVE='-1')
subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/tmp/ollama.log', 'w'),
                 stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get('http://127.0.0.1:11434/api/version', timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull nomic-embed-text
for m in MODELS:
    !ollama pull {m}
for m in MODELS:     # load each now, so the second goes to the free GPU
    requests.post('http://127.0.0.1:11434/api/chat', timeout=600, json={
        'model': m, 'messages': [{'role': 'user', 'content': 'hi'}], 'stream': False,
        'options': {'num_ctx': 16384}}).raise_for_status()
!ollama ps

## 3. Download SWE-bench Lite

In [ ]:
!python scripts/swebench.py download --split lite

## 4. Run both models

Each model runs in its own process with its own copies of the repositories.
This cell prints progress every 10 minutes until both finish.

In [ ]:
import re, shutil
selection = {'full': '', 'heldout': '--exclude-sample 60 --exclude-seed 0',
             'pilot': '--repos psf/requests'}[MODE]
if SAMPLE:
    selection += f' --sample {SAMPLE} --seed 1'

procs = {}
for i, m in enumerate(MODELS):
    tag = re.sub(r'[^A-Za-z0-9.]+', '-', m)
    if m in RESUME:
        dst = os.path.join(RUNS, os.path.basename(RESUME[m].rstrip('/')))
        if not os.path.exists(dst):
            shutil.copytree(RESUME[m], dst)      # /kaggle/input is read-only
        cmd = f'python scripts/agent_eval.py run --resume {dst}'
    else:
        cmd = f'python scripts/agent_eval.py run --model {m} {selection}'
    cmd += (f' --embed --checkouts /tmp/checkouts-{tag} --out {RUNS}'
            f' --max-hours {MAX_HOURS}')
    print(cmd)
    procs[m] = (subprocess.Popen(cmd, shell=True, stdout=open(f'/tmp/{tag}.log', 'w'),
                                 stderr=subprocess.STDOUT), f'/tmp/{tag}.log')

while any(p.poll() is None for p, _ in procs.values()):
    time.sleep(600)
    for m, (p, log) in procs.items():
        lines = open(log, encoding='utf-8', errors='replace').read().splitlines()
        tasks = [l for l in lines if l.startswith('[')]
        print(f'{time.strftime("%H:%M")} {m}: {tasks[-1] if tasks else "starting"}'
              f'{"  (finished)" if p.poll() is not None else ""}', flush=True)

for m, (p, log) in procs.items():
    print(f'\n===== {m} (exit {p.returncode}) - last lines of its log')
    print('\n'.join(open(log, encoding='utf-8', errors='replace').read().splitlines()[-60:]))

## 5. Results and download

In [ ]:
import glob
runs = sorted(glob.glob(f'{RUNS}/*-agent-*'))
if runs:
    !python scripts/agent_eval.py report {' '.join(runs)}
for _, log in procs.values():
    shutil.copy(log, RUNS)                       # keep the logs with the results
shutil.make_archive('/kaggle/working/codeorbit-agent-runs', 'zip', RUNS)
print('download /kaggle/working/codeorbit-agent-runs.zip from the Output tab')